# Homework: Evaluation

## Preparations

In [163]:
import json

from gitsource import GithubRepositoryDataReader
from pydantic import BaseModel
from dotenv import load_dotenv
from anthropic import Anthropic

from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

from evaluation_utils import llm_structured, llm_structured_retry

import pandas as pd
import numpy as np

from gitsource import chunk_documents

from ingest import load_faq_data, build_text_index, build_vector_index

from sentence_transformers import SentenceTransformer

from functools import partial

In [ ]:
#importlib.reload(ingest)

<module 'ingest' from '/Users/fakrueg/projects/courses/datatalks/llm-zoomcamp/llm-zoomcamp-homework/04-evaluation/ingest.py'>

In [2]:
# load environment variables
load_dotenv()

True

In [40]:
# load documents rag knowledge base
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

# check if the expected number of 72 documents was loaded
print(
    f"Number of documents loaded: {len(documents)}\n"
    f"Loaded expected number of documents: {len(documents) == 72}"
)

Number of documents loaded: 72
Loaded expected number of documents: True


In [4]:
# instructions for generating ground truth questions
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

## Exercise 1

In [5]:
# as specified in the exercise, work on a subset of the documents first
# use only the first three entries
documents_subset = documents[:3]
print("First three documents to work on for exercise 1:")
print(json.dumps(documents_subset, indent=2))

First three documents to work on for exercise 1:
[
  {
    "content": "# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we'll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type \"how are\" in WhatsApp, it suggests\n\"you\" as the next word. \"How are

In [6]:
# make pydantic model to get structured output from model
class Questions(BaseModel):
    questions: list[str]

In [7]:
# initialize Anthropic client
anthropic = Anthropic()

In [8]:
# call an LLM to get 5 questions for a document
# use only the first document for now to test behavior before iterating
# use the predefined function from the provided utils script
# the task is defined in the instructions (data_gen_instructions)
# the user prompt is the subset of documents to work on
questions_obj, usage = llm_structured(
    client=Anthropic(),
    instructions=data_gen_instructions,
    user_prompt=json.dumps(documents_subset[0]),
    output_type=Questions,
)

In [9]:
# have a look at what the LLM returned
# since I'm doing this to learn how to use and implement things,
# I'll try different ways of accessing the data

# first: attribute access
# the pydantic Questions model has one attibute: questions
questions_obj.questions

['What exactly is a large language model and how does it work at a basic level?',
 'What are the main problems with LLMs that RAG is designed to address?',
 'How does RAG help an LLM provide better answers to questions about specific documents or data?',
 'What will we build in this module and why is it useful as a practical example?',
 'Why does the course start with plain Python instead of jumping straight to frameworks?']

In [10]:
# next way of accessing the questions: Dict/JSON-style access via pydantic
questions_obj.model_dump()

{'questions': ['What exactly is a large language model and how does it work at a basic level?',
  'What are the main problems with LLMs that RAG is designed to address?',
  'How does RAG help an LLM provide better answers to questions about specific documents or data?',
  'What will we build in this module and why is it useful as a practical example?',
  'Why does the course start with plain Python instead of jumping straight to frameworks?']}

In [11]:
# the question also asks about the number of input tokens
# see how to access them
print(f"input tokens: {usage.input_tokens}")

# just out of curiosity: also check number of output tokens
print(f"output tokens: {usage.output_tokens}")

input tokens: 1300
output tokens: 102


In [12]:
# now generate questions for the three pages as specified in the exercise
questions_subset = []
usages = []

# iterate through the three pages, generate questions and collect usage
for doc in documents_subset:
    q, u = llm_structured(
        client=Anthropic(),
        instructions=data_gen_instructions,
        user_prompt=json.dumps(doc),
        output_type=Questions,
    )
    questions_subset.append(q)
    usages.append(u)

In [13]:
# check results
# first, have a look at the generated questions
print("-" * 100)
for q_set in questions_subset:
    for q in q_set.questions:
        print(q)
    print("-" * 100)

----------------------------------------------------------------------------------------------------
What is a large language model and how does it work at a basic level?
What are the main limitations of LLMs that make them unreliable for certain tasks?
How does RAG address the problems that LLMs have with knowledge cutoff and hallucinations?
What specific project are you going to build in this course module?
What's the difference between what you'll learn in Part 1 versus Part 2 of this module?
----------------------------------------------------------------------------------------------------
What do I need to install before I can start working on this course?
How should I securely store my OpenAI API key so it doesn't get accidentally uploaded to version control?
What's the difference between using uv versus pip for managing Python packages in this course?
How do I set up the OpenAI client to work with alternative providers like Groq instead of using OpenAI directly?
What command sh

In [24]:
# now check the usage
# inspect manually
print("Individual input tokens:")
for u in usages:
    print(u.input_tokens)

testlist = [1,2,3]
# compute average
print(
    "\nAverage input tokens:",
    np.mean([u.input_tokens for u in usages])
)

Individual input tokens:
1300
1688
2197

Average input tokens: 1728.3333333333333


## Prepare the next exercises

To save us some tokens, they prepared the ground-truth questions for us already.
I downloaded it like this:

```bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
wget ${PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv
```

Now load it into list of records called `ground_truth` with pandas.

In [32]:
# get pre-processed ground truth into pandas
df_ground_truth = pd.read_csv("ground-truth.csv")

# turn it into a list of dictionaries
ground_truth = df_ground_truth.to_dict(orient="records")

# check first few entries
# there should always be 5 questions per document
ground_truth[:12]

[{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'Why does this course build the RAG project in plain Python instead of starting with a framework or library?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What are the main weaknesses of large language models that this module is trying to work around?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What will the course build in the first part of the module, and how is the second part different?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What kind of example app are you building here, and what data will it answer questions from?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What do I need installed before starting this module?',
  'filename': '01-agentic-rag/lessons/02-environmen

In [43]:
# chunk documents for better retrieval
chunks = chunk_documents(documents, size=2000, step=1000)

# check results of chunking
print(f"Number of chunks: {len(chunks)}")
chunks[0:3]

Number of chunks: 295


[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [126]:
# build text search index
text_index = build_text_index(chunks)

# test text search index without filtering
text_index.search(
    query="LLM AI large language model",
    num_results=10,
)

[{'start': 0,
  'content': "# AI Orchestration\n\nVideo: [Using AI in Workflows](https://youtu.be/l9w6oeUtrpk)\n\nThis module introduces AI-powered workflow orchestration using Kestra. You'll learn how AI can accelerate workflow development and enable autonomous task automation through three techniques: AI Copilot for generating flows, RAG for grounding responses in real data, and AI Agents for autonomous task execution.\n\n## Learning Objectives\n\nBy the end of this module, you will:\n\n- Understand the importance of context engineering in AI applications\n- Use AI Copilot to build Kestra flows faster and more accurately\n- Implement Retrieval Augmented Generation (RAG) to ground AI responses in real data\n- Build autonomous AI agents that can make decisions and use tools dynamically\n- Design multi-agent systems where specialized agents collaborate to solve complex tasks\n- Apply best practices for using AI in production data workflows\n\n## Prerequisites\n\n- Kestra running locally

In [127]:
# test text search index with filtering
text_index.search(
    query="LLM AI large language model",
    filter_dict={"filename": "01-agentic-rag/lessons/01-intro.md"},
    num_results=10,
)

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [130]:
# wrap it into a function
def text_search(query, filterdict=None, num_results=5):
    
    return text_index.search(
        query=query,
        filter_dict=filterdict,
        num_results=num_results,
    )

# test the function without filtering
text_search(query="engineering", num_results=10)

[{'start': 2000,
  'content': " `data-engineering-zoomcamp`, etc. We'll\nuse these slugs for filtering in search.\n\n## Using this data\n\nIn the RAG pipeline, this dataset is our knowledge base:\n\n1. We index all the documents (the search step)\n2. When a student asks a question, we search the index\n3. The search returns the most relevant FAQ entries\n4. We give those entries to the LLM as context\n5. The LLM generates an answer based on the context\n\nThe `question` and `answer` fields contain the text we'll search\nthrough. The `course` field lets us filter by course. For example, if a\nstudent asks about the data engineering course, we skip results from\nthe ML course. The `section` field helps with ranking - knowing which\npart of the course a question belongs to is useful context.\n\n## A note on data preparation\n\nIn our case, the data is already prepared. I maintain this FAQ website\nand made sure the data comes back in a convenient JSON format. So we\ndon't need to do much 

In [131]:
# test the function with filtering
text_search(
    query="engineering",
    filterdict={"filename": "01-agentic-rag/lessons/01-intro.md"},
    num_results=10
)

[]

In [132]:
# also build a vector search index
vector_index = build_vector_index(chunks)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

In [135]:
# initialize model for embedding queries
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [136]:
# define function for searching the vector index
def vector_search(
    query,
    filterdict=None,
    model=model,
    vindex=vector_index,
    num_results=5
):
    query_vector = model.encode(query)
    return vindex.search(
        query_vector=query_vector,
        filter_dict=filterdict,
        num_results=num_results,
    )

In [137]:
# test vector search index without filtering
vector_search(query="What is an LLM?", num_results=10)

[{'start': 1000,
  'content': 'the next\nword based on what you typed so far.\n\nA large language model does the same thing, but at a much larger scale.\nIt has billions of parameters and is trained on most of the text on the\ninternet. When it predicts the next word, it feels like you\'re talking\nto an intelligent being. It understands what you ask and gives\nmeaningful answers.\n\nIn this course, we treat LLMs as black boxes. We won\'t look inside or\ncover the theory, and we won\'t host a model ourselves. We use an LLM\nprovider and call it over an API. For us, an LLM is a box: text goes in,\ntext comes out.\n\nBut LLMs have limitations:\n\n- Knowledge cutoff: they only know what was in their training data.\n  If you ask about something that happened after training, they won\'t\n  know - or worse, they\'ll make something up.\n- No access to your data: they can\'t see your documents, databases,\n  or internal systems unless you provide that information.\n- Hallucinations: they somet

In [138]:
# test vector search index with filtering
vector_search(
    query="What is an LLM?",
    filterdict={"filename": "01-agentic-rag/lessons/01-intro.md"},
    num_results=10
)

[{'start': 1000,
  'content': 'the next\nword based on what you typed so far.\n\nA large language model does the same thing, but at a much larger scale.\nIt has billions of parameters and is trained on most of the text on the\ninternet. When it predicts the next word, it feels like you\'re talking\nto an intelligent being. It understands what you ask and gives\nmeaningful answers.\n\nIn this course, we treat LLMs as black boxes. We won\'t look inside or\ncover the theory, and we won\'t host a model ourselves. We use an LLM\nprovider and call it over an API. For us, an LLM is a box: text goes in,\ntext comes out.\n\nBut LLMs have limitations:\n\n- Knowledge cutoff: they only know what was in their training data.\n  If you ask about something that happened after training, they won\'t\n  know - or worse, they\'ll make something up.\n- No access to your data: they can\'t see your documents, databases,\n  or internal systems unless you provide that information.\n- Hallucinations: they somet

In [140]:
# use provided functions for RFF and hybrid search

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, filterdict=None, k=60):
    text_results = text_search(query, filterdict=filterdict, num_results=10)
    vector_results = vector_search(query, filterdict=filterdict, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [141]:
# test hybrid search function
hybrid_search(query="What is an LLM?")

[{'start': 4000,
  'content': "hat the LLM generates.\nThat search step is what gives the LLM the context it needs to answer\ncorrectly.\n\nWhat we just did was naive. I knew in advance which FAQ entry held the\nanswer and pasted it in by hand. What we want instead is to perform\nsearch automatically. We take the student's question, find the most\nrelevant documents, and send those to the LLM.\n\nIn code, it looks like this:\n\n```python\ndef rag(question):\n    search_results = search(question)\n    user_prompt = build_prompt(question, search_results)\n    return llm(user_prompt)\n```\n\nThat's the entire architecture. It comes down to three components.\n\nThe pieces are search, the prompt, and the LLM:\n\n- search\n- prompt\n- LLM\n\n\n```mermaid\nflowchart TD\n    U([User])\n\n    APP[Application]\n\n    DB[(DB)]\n    DOCS[[D1 ... D5]]\n\n    PROMPT[Build Prompt<br/>Question + Context]\n\n    LLM[LLM]\n\n    ANSWER([Answer])\n\n    U -->|Question| APP\n\n    APP -->|Query| DB\n    D

## Exercise 2

In [ ]:
# save first question from ground truth to object
q = ground_truth[0]["question"]
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [145]:
# run text search on the first question
text_search(q)

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

## Exercise 3

In [147]:
# run vector search on the same question
vector_search(q)

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

This question was generated from `01-agentic-rag/lessons/01-intro.md`.
Both found it in the top 5 results, but only vector search determined it as the
best match.

In [ ]:
# out of curiosity, now check hybrid search as well
hybrid_search(q)

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

Hybrid search also identified the right one.

## Exercise 4

In [153]:
# define functions for calculating search metrics

def compute_relevance(q, search_function):
    filename = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == filename))

    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [154]:
# evaluate text search
evaluate(
    ground_truth=ground_truth,
    search_function=text_search,
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

## Exercise 5

In [155]:
# evaluate vector search
evaluate(
    ground_truth=ground_truth,
    search_function=vector_search,
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.8083333333333333, 'mrr': 0.6356944444444446}

## Exercise 6

In [169]:
# evaluate hybrid search on different values for k

# define list of k values to evaluate
ks = [1, 50, 100, 200]

# initialize dictionary for holding results
results = {}

# iterate through k values and evaluate hybrid search
for k in ks:
    results[k] = evaluate(
        ground_truth=ground_truth,
        search_function=partial(hybrid_search, k=k),
    )

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

In [170]:
# print results
for k, v in results.items():
    print(f"k={k}: {v}")

k=1: {'hit_rate': 0.8583333333333333, 'mrr': 0.6722685185185188}
k=50: {'hit_rate': 0.8472222222222222, 'mrr': 0.6721296296296295}
k=100: {'hit_rate': 0.8472222222222222, 'mrr': 0.6721296296296295}
k=200: {'hit_rate': 0.8472222222222222, 'mrr': 0.6721296296296295}
